In [ ]:
! pip install evaluate

In [1]:
import numpy as np

In [2]:
from transformers import pipeline

In [ ]:
import torch
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import DataCollatorWithPadding
from transformers import TrainingArguments, Trainer

from datasets import load_dataset
import evaluate

## Fine-tuning a pre-trained model (Trainer API)
Fine tuning a BERT model for text classification

In [6]:
# Pre-trained model to be used
checkpoint = 'bert-base-uncased'

In [7]:
# Tokenize the training data
# Dataset: Microsoft Research Paraphrase Corpus (MRPC) dataset, Link to paper: https://aclanthology.org/I05-5002.pdf)
# The dataset contains 5801 sentence pairs each labelled manually if one is a paraphrase of the other

raw_datasets = load_dataset('glue', 'mrpc')
raw_datasets

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

mrpc/train-00000-of-00001.parquet:   0%|          | 0.00/649k [00:00<?, ?B/s]

mrpc/validation-00000-of-00001.parquet:   0%|          | 0.00/75.7k [00:00<?, ?B/s]

mrpc/test-00000-of-00001.parquet:   0%|          | 0.00/308k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [6]:
# Preview a pair in the dataset
raw_datasets["train"][11]

{'sentence1': 'The Nasdaq composite index increased 10.73 , or 0.7 percent , to 1,514.77 .',
 'sentence2': 'The Nasdaq Composite index , full of technology stocks , was lately up around 18 points .',
 'label': 0,
 'idx': 12}

In [7]:
# Check dataset features/labels
raw_datasets['train'].features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32')}

### Data preprocessing

In [8]:
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
# Function to aid tokenizing (since the dataset is within a dictionary)
def tokenize_function(example):
    """
    Input: takes a dictionary (items of the dataset)
    Output: a dictionary (with keys sentence 1/2, label, ids, attn mask etc.)
    """
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

In [10]:
# Apply tokenization to the dataset
tokenized_dataset = raw_datasets.map(tokenize_function, batched = True)

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

In [10]:
# inspect
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

In [11]:
# Dynamic padding
# We apply padding within each individual batch (more efficient as we only pad according to the lonest sequence within that particular batch)
# To be used later in the training step

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

### Training & Evaluation

In [12]:
# Evaluation function (to help return metrics during training process)

def compute_metrics(eval_preds):
    """
    Args
        eval_preds : predictions (logits) from the model
    """
    metric = evaluate.load("glue", "mrpc")
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis = -1) # Convert logits to binary predictions

    return metric.compute(predictions = predictions, references = labels)

In [ ]:
# Define training arguments
training_args = TrainingArguments(output_dir = "test-trainer", 
                                  eval_strategy = "epoch",
                                  fp16 = True, # enables mixed-precision for faster training and reduced memory usage
                                  
                                  )

# Define the model
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels = 2)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
# Define the trainer
trainer = Trainer(
    model,
    training_args,
    train_dataset = tokenized_dataset["train"],
    eval_dataset = tokenized_dataset["validation"],
    data_collator = data_collator,
    processing_class = tokenizer,
    compute_metrics = compute_metrics
)

In [15]:
# Train and evaluate
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.455743,0.803922,0.873016
2,0.536719,0.417402,0.848039,0.892361
3,0.327971,0.705699,0.850490,0.898164


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1377, training_loss=0.3608114482184294, metrics={'train_runtime': 130.1133, 'train_samples_per_second': 84.572, 'train_steps_per_second': 10.583, 'total_flos': 405114969714960.0, 'train_loss': 0.3608114482184294, 'epoch': 3.0})

## Fine-tuning a pre-trained model (from scratch with PyTorch)

In [ ]:
## TBD later

## Fine-tuning an LLM with LoRA